In [1]:
import pandas as pd
import numpy as np
import sqlite3

# Load all six CSV files from the data/ directory
movies = pd.read_csv('data/movies.csv')
genres = pd.read_csv('data/genres.csv')
movie_genres = pd.read_csv('data/movie_genres.csv')
cast = pd.read_csv('data/cast.csv')
crew = pd.read_csv('data/crew.csv')
movie_keywords = pd.read_csv('data/movie_keywords.csv')

tables = {
    'movies': movies,
    'genres': genres,
    'movie_genres': movie_genres,
    'cast': cast,
    'crew': crew,
    'movie_keywords': movie_keywords
}

# Inspect shape, data types, and null counts for every table
for name, df in tables.items():
    print(f"================ {name.upper()} ================")
    print(f"Shape: {df.shape}")
    print("\nData Types & Null Counts:")
    summary = pd.DataFrame({'dtype': df.dtypes, 'null_count': df.isnull().sum()})
    print(summary)
    print("\n")


================ MOVIES ================
Shape: (2509, 12)

Data Types & Null Counts:
                     dtype  null_count
movie_id             int64           0
title                  str           0
original_language      str           0
release_date           str           0
runtime              int64           0
budget             float64           0
revenue            float64           0
popularity         float64           0
vote_average       float64           0
vote_count           int64           0
status                 str           0
overview               str           0


================ GENRES ================
Shape: (19, 2)

Data Types & Null Counts:
            dtype  null_count
genre_id    int64           0
genre_name    str           0


================ MOVIE_GENRES ================
Shape: (6888, 2)

Data Types & Null Counts:
          dtype  null_count
movie_id  int64           0
genre_id  int64           0


================ CAST ================
Shape: (24917,

In [2]:
# 1. CLEAN MOVIES TABLE
movies = movies.drop_duplicates(subset=['movie_id']).copy()
movies['movie_id'] = pd.to_numeric(movies['movie_id'], errors='coerce').dropna().astype(int)
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Fix numeric columns while keeping 0s intact (0 = unknown in TMDB)
movies['runtime'] = pd.to_numeric(movies['runtime'], errors='coerce').fillna(0).astype(int)
movies['budget'] = pd.to_numeric(movies['budget'], errors='coerce').fillna(0).astype('int64')
movies['revenue'] = pd.to_numeric(movies['revenue'], errors='coerce').fillna(0).astype('int64')
movies['popularity'] = pd.to_numeric(movies['popularity'], errors='coerce').fillna(0.0).astype(float)
movies['vote_average'] = pd.to_numeric(movies['vote_average'], errors='coerce').fillna(0.0).astype(float)
movies['vote_count'] = pd.to_numeric(movies['vote_count'], errors='coerce').fillna(0).astype(int)

# Fill text nulls safely
movies['title'] = movies['title'].fillna('Unknown Title').str.strip()
movies['original_language'] = movies['original_language'].fillna('unknown').str.strip()
movies['status'] = movies['status'].fillna('Unknown').str.strip()
movies['overview'] = movies['overview'].fillna('')

valid_movie_ids = set(movies['movie_id'])

# 2. CLEAN GENRES TABLE
genres = genres.drop_duplicates(subset=['genre_id']).copy()
genres['genre_id'] = genres['genre_id'].astype(int)
genres['genre_name'] = genres['genre_name'].str.strip()
valid_genre_ids = set(genres['genre_id'])

# 3. CLEAN MOVIE_GENRES TABLE (Remove duplicates & orphaned FKs)
movie_genres = movie_genres.drop_duplicates(subset=['movie_id', 'genre_id']).copy()
movie_genres['movie_id'] = movie_genres['movie_id'].astype(int)
movie_genres['genre_id'] = movie_genres['genre_id'].astype(int)
movie_genres = movie_genres[
    movie_genres['movie_id'].isin(valid_movie_ids) & 
    movie_genres['genre_id'].isin(valid_genre_ids)
]

# 4. CLEAN CAST TABLE (Remove exact duplicates & orphaned movie_ids)
cast = cast.drop_duplicates().copy()
cast['movie_id'] = cast['movie_id'].astype(int)
cast['person_id'] = cast['person_id'].astype(int)
cast['cast_order'] = pd.to_numeric(cast['cast_order'], errors='coerce').fillna(999).astype(int)
cast['actor_name'] = cast['actor_name'].fillna('Unknown').str.strip()
cast['character_name'] = cast['character_name'].fillna('Unknown').str.strip()
cast = cast[cast['movie_id'].isin(valid_movie_ids)].reset_index(drop=True)
if 'cast_row_id' not in cast.columns:
    cast.insert(0, 'cast_row_id', range(1, len(cast) + 1))

# 5. CLEAN CREW TABLE (Remove exact duplicates & orphaned movie_ids)
crew = crew.drop_duplicates().copy()
crew['movie_id'] = crew['movie_id'].astype(int)
crew['person_id'] = crew['person_id'].astype(int)
crew['person_name'] = crew['person_name'].fillna('Unknown').str.strip()
crew['job'] = crew['job'].fillna('Unknown').str.strip()
crew['department'] = crew['department'].fillna('Unknown').str.strip()
crew = crew[crew['movie_id'].isin(valid_movie_ids)].reset_index(drop=True)
if 'crew_row_id' not in crew.columns:
    crew.insert(0, 'crew_row_id', range(1, len(crew) + 1))

# 6. CLEAN MOVIE_KEYWORDS TABLE (Remove exact duplicates & orphaned movie_ids)
movie_keywords = movie_keywords.drop_duplicates(subset=['movie_id', 'keyword_id']).copy()
movie_keywords['movie_id'] = movie_keywords['movie_id'].astype(int)
movie_keywords['keyword_id'] = movie_keywords['keyword_id'].astype(int)
movie_keywords['keyword_name'] = movie_keywords['keyword_name'].fillna('').str.strip()
movie_keywords = movie_keywords[movie_keywords['movie_id'].isin(valid_movie_ids)].reset_index(drop=True)

print("All tables cleaned successfully!")


All tables cleaned successfully!


In [3]:
conn = sqlite3.connect('tmdb_movies.db')

# Write all 6 cleaned DataFrames to SQLite tables
movies.to_sql('movies', conn, if_exists='replace', index=False)
genres.to_sql('genres', conn, if_exists='replace', index=False)
movie_genres.to_sql('movie_genres', conn, if_exists='replace', index=False)
cast.to_sql('cast', conn, if_exists='replace', index=False)
crew.to_sql('crew', conn, if_exists='replace', index=False)
movie_keywords.to_sql('movie_keywords', conn, if_exists='replace', index=False)

# Verification test query across movies, movie_genres, and genres
test_query = """
SELECT m.title, m.release_date, g.genre_name
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
LIMIT 10;
"""
print(pd.read_sql_query(test_query, conn))
conn.close()


             title release_date       genre_name
0     Interstellar   2014-11-05        Adventure
1     Interstellar   2014-11-05            Drama
2     Interstellar   2014-11-05  Science Fiction
3        Inception   2010-07-15        Adventure
4        Inception   2010-07-15           Action
5        Inception   2010-07-15  Science Fiction
6     The Avengers   2012-04-25        Adventure
7     The Avengers   2012-04-25           Action
8     The Avengers   2012-04-25  Science Fiction
9  The Dark Knight   2008-07-16           Action
